# Latente rum: hvordan en model gemmer "mening" som tal

Velkommen til forberedelsen til transformer-workshoppen!

I autoencoder-emnet så I, at et netværk kunne presse et billede ned til få tal — en **vektor** — og at
*ens* ting endte *tæt* på hinanden i det, vi kaldte det **latente rum**. En transformer (modellen bag fx
ChatGPT) bruger præcis samme idé, bare på **tekst**: hvert tegn bliver lavet om til en vektor, før modellen
regner videre. I denne notebook bygger vi broen fra autoencoder til transformer og kigger direkte ind i den
færdige models latente rum.

> Opgaverne er mærket med **(ekstra)** for de svære og **(find fejlen)** for fejlfindings-opgaver.
> Nogle opgaver skal I bare *tænke* over og diskutere — ikke skrive svaret ned.

## Setup

In [ ]:
# Henter den færdige transformer-model fra GitHub (Plan B: upload base_model.pth via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/27-Models/base_model.pth

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print("Klar!")

# 1: Fra autoencoder til latent rum

Husk autoencoderen: den pressede 784 pixels ned til fx 2 tal og kunne bygge billedet op igen. De 2 tal var
et **punkt** i et latent rum, og cifre der lignede hinanden, lå tæt på hinanden — helt af sig selv.

Den samme idé driver sprogmodeller: et ord (eller tegn) beskrives med en **vektor** (en liste af tal), og
"betydning" bliver til **placering**. Lad os få en fornemmelse med nogle vektorer, vi selv finder på.

In [ ]:
# Vi opdigter selv nogle 2D-"betydningsvektorer". (I virkeligheden LÆRER modellen dem —
# her sætter vi dem bare selv for at få en fornemmelse.)
word_vectors = {
    "konge":    torch.tensor([2.0, 3.0]),
    "dronning": torch.tensor([2.2, 2.8]),
    "prins":    torch.tensor([1.8, 2.6]),
    "hund":     torch.tensor([-2.0, -1.0]),
    "kat":      torch.tensor([-1.7, -1.3]),
    "bil":      torch.tensor([3.0, -3.0]),
}

plt.figure(figsize=(6, 5))
for word, v in word_vectors.items():
    plt.scatter(v[0], v[1])
    plt.annotate(word, (v[0], v[1]), fontsize=12)
plt.axhline(0, color="gray", lw=0.5)
plt.axvline(0, color="gray", lw=0.5)
plt.title("Ord som punkter i et (opdigtet) latent rum")
plt.show()

Læg mærke til: **konge**, **dronning** og **prins** klumper sammen (kongelige), mens **hund** og
**kat** ligger for sig (kæledyr). Retningen og afstanden bærer altså en slags mening.

### Opgaver

##### Opgave 1.1
Kig på plottet. Hvorfor giver det mening, at **hund** og **kat** ligger tæt på hinanden, men langt fra **bil**?
Hvad ville det betyde, hvis to ords vektorer lå oven i hinanden?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Giv ordet `"prinsesse"` en vektor, og placér den, så den lander tæt på **dronning**. Kør cellen og se, hvor den havner.

In [ ]:
word_vectors["prinsesse"] = torch.tensor([...])   # ← vælg to tal (prøv tæt på "dronning": [2.2, 2.8])

plt.figure(figsize=(6, 5))
for word, v in word_vectors.items():
    plt.scatter(v[0], v[1])
    plt.annotate(word, (v[0], v[1]), fontsize=12)
plt.title("Med dit eget ord")
plt.show()

# 2: Ord og tegn som vektorer — `nn.Embedding`

Hvor kommer vektorerne fra i en rigtig model? Fra et **opslagsbord** kaldet en *embedding*: hver token (her:
hvert tegn) har et nummer, og embedding-bordet slår nummeret op og giver en vektor. Det er det **allerførste**,
en transformer gør ved din tekst.

In [ ]:
# En embedding-tabel: 6 tokens, hver bliver til en vektor med 4 tal.
embedding = nn.Embedding(num_embeddings=6, embedding_dim=4)

token_ids = torch.tensor([0, 3, 5])     # tre tokens
vectors = embedding(token_ids)          # slå deres vektorer op

print("Token-id'er:", token_ids.tolist())
print("Deres vektorer:\n", vectors)
print("Form:", tuple(vectors.shape), "= (3 tokens, hver med 4 tal)")

Lige nu er tallene **tilfældige** — bordet er ikke trænet endnu. Under træning skubbes vektorerne
rundt, indtil "mening" ligger i geometrien. I næste afsnit kigger vi på et bord, der ER færdigtrænet.

### Opgaver

##### Opgave 2.1
Lav en embedding-tabel med **10 tokens**, hvor hver token bliver til en vektor med **16 tal**. Slå token nr. 3 op.

In [ ]:
my_embedding = nn.Embedding(num_embeddings=10, embedding_dim=...)   # ← udfyld tallet

vector3 = my_embedding(torch.tensor(3))
print("Vektor for token 3:", vector3)
print("Form:", tuple(vector3.shape))

##### Opgave 2.2
Et embedding-bord med 84 tegn og 384 tal pr. tegn har 84 × 384 ≈ 32.000 tal, som modellen selv skal lære.
Hvorfor kan man ikke bare bruge tegnets nummer (0, 1, 2, …) direkte i stedet for en hel vektor?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 3: Kig ind i den TRÆNEDE models latente rum

Nu bruger vi ikke et tilfældigt bord — vi henter det **rigtige** embedding-bord ud af den færdige
transformer (`base_model.pth`), som I skal lege med i workshoppen. Modellen arbejder på tegn, og den har
et fast vokabular på 84 tegn.

In [ ]:
# Det FASTE vokabular fra transformeren (samme 84 tegn som i workshoppen).
VOCAB_CHARS = (
    "\n "                            # linjeskift og mellemrum
    "abcdefghijklmnopqrstuvwxyz"     # små bogstaver
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"     # store bogstaver
    "æøåÆØÅ"                          # danske bogstaver
    "0123456789"                     # tal
    ".,!?:;-'\"*()[]"                # tegnsætning
)
chars = sorted(set(VOCAB_CHARS))     # samme rækkefølge som modellen bruger
print("Antal tegn i vokabularet:", len(chars))

# Hent embedding-bordet ud af den trænede model.
state = torch.load("base_model.pth", map_location="cpu")
E = state["transformer.wte.weight"]      # form (84, 384): én vektor pr. tegn
print("Embedding-bord:", tuple(E.shape), "= (84 tegn, 384 tal hver)")

384 tal pr. tegn kan vi ikke plotte direkte. Vi projicerer ned til **2D** med PCA — det beholder de
to "vigtigste" retninger i dataen (samme grundidé som autoencoderens flaskehals).

In [ ]:
# Projicér de 384-dimensionelle vektorer ned til 2 dimensioner med PCA.
E_centered = E - E.mean(dim=0)
U, S, V = torch.pca_lowrank(E_centered, q=2)
coords = E_centered @ V[:, :2]           # form (84, 2)

def nice(ch):
    return {"\n": "\\n", " ": "␣"}.get(ch, ch)

plt.figure(figsize=(11, 9))
for i, ch in enumerate(chars):
    plt.scatter(coords[i, 0], coords[i, 1], color="steelblue")
    plt.annotate(nice(ch), (coords[i, 0], coords[i, 1]), fontsize=11)
plt.title("Transformerens tegn-embeddings projiceret til 2D")
plt.show()

Uden at nogen har fortalt den det, har modellen samlet tegn, der *opfører sig ens*: bogstaver ét
sted, cifre et andet, tegnsætning for sig. Den lærte det udelukkende ved at gætte det næste tegn millioner
af gange. **Det er dét latente rum, transformeren "tænker" i.**

Vi kan også måle nærhed direkte med **cosine similarity** (måler vinklen mellem to vektorer: 1 = samme
retning/meget ens, 0 = vinkelret, −1 = modsat).

In [ ]:
def cosine_similarity(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm())

# Vi trækker "gennemsnits-tegnet" fra først (samme centrering som i PCA'en ovenfor),
# så vi sammenligner tegnenes RETNING og ikke bare den fælles baggrund, alle tegn deler.
E_dir = E - E.mean(dim=0)

def nearest_chars(ch, n=5):
    # Finder de n tegn, hvis retning ligner ch's mest (cosine similarity).
    i = chars.index(ch)
    sims = [(cosine_similarity(E_dir[i], E_dir[j]).item(), other)
            for j, other in enumerate(chars) if j != i]
    sims.sort(reverse=True)
    return [(round(s, 2), c) for s, c in sims[:n]]

print("Tegn der ligner 'a' mest:", nearest_chars("a"))
print("Tegn der ligner 't' mest:", nearest_chars("t"))
print("Tegn der ligner '5' mest:", nearest_chars("5"))

Læs outputtet: **a** ligger tættest på andre **vokaler** (e, o, i), og **t** tæt på andre almindelige
**konsonanter** (r, n, d). Men **5** havner sammen med sjældne tegn som Ø, Æ og q — ikke fordi et 5-tal
"ligner" et ø, men fordi modellen **næsten aldrig har set dem** i fantasy-teksten og derfor ikke har lært
dem ordentligt endnu. Det latente rum afslører altså også, hvad modellen *ikke* har øvet sig på!

### Opgaver

##### Opgave 3.1
Færdiggør `my_cosine`. Cosine similarity er prikproduktet af to vektorer **divideret med** deres længder
ganget sammen. Tip: en vektors længde er `v.norm()`.

In [ ]:
def my_cosine(a, b):
    return torch.dot(a, b) / (...)   # ← divider med (a's længde gange b's længde)

# Test — skal give (næsten) det samme som torch's egen:
print("min:  ", round(my_cosine(E[0], E[1]).item(), 4))
print("torch:", round(torch.nn.functional.cosine_similarity(E[0], E[1], dim=0).item(), 4))

##### Opgave 3.2
Skift tegnet ud og find dets nærmeste naboer i det latente rum. Passer naboerne med din intuition?

In [ ]:
mit_tegn = "q"   # ← prøv fx "æ", "7", "!" eller et mellemrum " "
print(f"Tegn der ligner {mit_tegn!r} mest:", nearest_chars(mit_tegn))

##### Opgave 3.3 (ekstra)
Plot **kun vokalerne** oven i hinanden og se, om de klumper sammen. Prøv bagefter at lave det samme plot
for cifrene `"0123456789"`.

In [ ]:
vokaler = list("aeiouyæøå")

plt.figure(figsize=(7, 6))
for ch in vokaler:
    i = chars.index(ch)
    plt.scatter(coords[i, 0], coords[i, 1], color="crimson")
    plt.annotate(ch, (coords[i, 0], coords[i, 1]), fontsize=14)
plt.title("Kun vokaler i det latente rum")
plt.show()

##### Opgave 3.4
Vi har set, at hvert tegn bliver til en vektor i et latent rum, hvor "ens" tegn ligger tæt. Det er kun det
**første** skridt i en transformer — bagefter blander den vektorerne sammen med *attention*. Hvorfor tror I,
det er en god start at have tegn med samme rolle placeret tæt på hinanden, før den regner videre?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*